# Week 6 – Decision Trees and Random Forests

## Objective

This notebook applies decision trees and random forests to the NBA game total prediction project.

The goal is to predict whether an NBA game finishes over or under the sportsbook's pregame total.

The analysis will examine:

- Decision Tree classification
- Tree depth and overfitting
- Feature importance
- Random Forest classification
- Comparison of Decision Tree and Random Forest performance

The binary target is:

- `1` = Actual game total is greater than the sportsbook total (Over)
- `0` = Actual game total is less than the sportsbook total (Under)

Games where the actual total exactly equals the sportsbook total are excluded as pushes.

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score

)

In [23]:
merged = pd.read_csv(
    "../data/processed/merged_games_betting.csv",
    low_memory=False
)

print("Dataset shape:", merged.shape)

Dataset shape: (21965, 54)


In [24]:
classification_df = merged[
    merged["actual_total"] != merged["total"]
].copy()

classification_df["over"] = (
    classification_df["actual_total"] > classification_df["total"]
).astype(int)

In [25]:
features = [
    "total",
    "spread",
    "moneyline_home",
    "moneyline_away",
    "season"
]

tree_df = classification_df[
    features + ["over"]
].dropna()

X = tree_df[features]
y = tree_df["over"]

print("Shape:", tree_df.shape)

print("\nClass balance:")
print(y.value_counts(normalize=True))

Shape: (18420, 6)

Class balance:
over
0    0.501466
1    0.498534
Name: proportion, dtype: float64


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (14736, 5)
X_test: (3684, 5)


## Decision Tree Classifier

In [27]:
tree_default = DecisionTreeClassifier(
    random_state=42
)

tree_default.fit(X_train, y_train)

y_pred_tree = tree_default.predict(X_test)
y_prob_tree = tree_default.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print("Precision:", precision_score(y_test, y_pred_tree))
print("Recall:", recall_score(y_test, y_pred_tree))
print("F1 Score:", f1_score(y_test, y_pred_tree))
print("ROC AUC:", roc_auc_score(y_test, y_prob_tree))

Accuracy: 0.49212812160694897
Precision: 0.4905555555555556
Recall: 0.48067501360914533
F1 Score: 0.4855650261204289
ROC AUC: 0.49031753886527285


In [28]:
train_pred_tree = tree_default.predict(X_train)

print(
    "Training Accuracy:",
    accuracy_score(y_train, train_pred_tree)
)

print(
    "Test Accuracy:",
    accuracy_score(y_test, y_pred_tree)
)

print(
    "Tree Depth:",
    tree_default.get_depth()
)

print(
    "Number of Leaves:",
    tree_default.get_n_leaves()
)

Training Accuracy: 0.9832383279044516
Test Accuracy: 0.49212812160694897
Tree Depth: 57
Number of Leaves: 6581


In [29]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

tree_cv = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid={
        "max_depth": [2, 3, 5, 8, 10, 15]
    },
    cv=cv,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1
)

tree_cv.fit(X_train, y_train)

best_tree = tree_cv.best_estimator_

y_pred_best_tree = best_tree.predict(X_test)
y_prob_best_tree = best_tree.predict_proba(X_test)[:, 1]

print("Best Decision Tree depth:", tree_cv.best_params_["max_depth"])
print("CV Accuracy:", tree_cv.best_score_)
print("Test Accuracy:", accuracy_score(y_test, y_pred_best_tree))

tree_depth_results = pd.DataFrame({
    "Depth": tree_cv.cv_results_["param_max_depth"],
    "Train Accuracy": tree_cv.cv_results_["mean_train_score"],
    "CV Accuracy": tree_cv.cv_results_["mean_test_score"]
})

tree_depth_results.round(4)

Best Decision Tree depth: 10
CV Accuracy: 0.5066503800217155
Test Accuracy: 0.5149294245385451


,Depth,Train Accuracy,CV Accuracy
0,2,0.5042,0.5020
1,3,0.5051,0.5011
2,5,0.5120,0.5039
3,8,0.5239,0.5041
4,10,0.5362,0.5067
5,15,0.5821,0.5035


max_depth 5 was best as shown above

In [30]:
best_tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

best_tree.fit(X_train, y_train)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_tree.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

importance_df

,Feature,Importance
0,total,0.380931
3,moneyline_away,0.370465
1,spread,0.109141
4,season,0.083162
2,moneyline_home,0.056301


## Random Forest

In [31]:
forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

forest.fit(X_train, y_train)

y_pred_forest = forest.predict(X_test)
y_prob_forest = forest.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_forest))
print("Precision:", precision_score(y_test, y_pred_forest))
print("Recall:", recall_score(y_test, y_pred_forest))
print("F1 Score:", f1_score(y_test, y_pred_forest))
print("ROC AUC:", roc_auc_score(y_test, y_prob_forest))

Accuracy: 0.5
Precision: 0.4985656913367757
Recall: 0.47305389221556887
F1 Score: 0.48547486033519555
ROC AUC: 0.49632177295259367


In [32]:
train_pred_forest = forest.predict(X_train)

print(
    "Training Accuracy:",
    accuracy_score(y_train, train_pred_forest)
)

print(
    "Test Accuracy:",
    accuracy_score(y_test, y_pred_forest)
)

Training Accuracy: 0.9832383279044516
Test Accuracy: 0.5


In [33]:
forest_cv = GridSearchCV(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    param_grid={
        "max_depth": [3, 5, 8, 10, 15]
    },
    cv=cv,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1
)

forest_cv.fit(X_train, y_train)

best_forest = forest_cv.best_estimator_

y_pred_best_forest = best_forest.predict(X_test)
y_prob_best_forest = best_forest.predict_proba(X_test)[:, 1]

print("Best Random Forest depth:", forest_cv.best_params_["max_depth"])
print("CV Accuracy:", forest_cv.best_score_)
print("Test Accuracy:", accuracy_score(y_test, y_pred_best_forest))

rf_depth_results = pd.DataFrame({
    "Depth": forest_cv.cv_results_["param_max_depth"],
    "Train Accuracy": forest_cv.cv_results_["mean_train_score"],
    "CV Accuracy": forest_cv.cv_results_["mean_test_score"]
})

rf_depth_results.round(4)

Best Random Forest depth: 15
CV Accuracy: 0.5103827361563518
Test Accuracy: 0.497557003257329


,Depth,Train Accuracy,CV Accuracy
0,3,0.5365,0.5026
1,5,0.5664,0.5036
2,8,0.6482,0.5061
3,10,0.7185,0.5082
4,15,0.8893,0.5104


In [34]:
best_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=42
)

best_forest.fit(X_train, y_train)

y_pred_best_forest = best_forest.predict(X_test)
y_prob_best_forest = best_forest.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_best_forest))
print("Precision:", precision_score(y_test, y_pred_best_forest))
print("Recall:", recall_score(y_test, y_pred_best_forest))
print("F1 Score:", f1_score(y_test, y_pred_best_forest))
print("ROC AUC:", roc_auc_score(y_test, y_prob_best_forest))

Accuracy: 0.5206297502714441
Precision: 0.5266316579144786
Recall: 0.38214480130647793
F1 Score: 0.44290220820189274
ROC AUC: 0.5237994258075374


In [35]:
best_tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

best_tree.fit(X_train, y_train)

y_pred_best_tree = best_tree.predict(X_test)
y_prob_best_tree = best_tree.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_best_tree))
print("Precision:", precision_score(y_test, y_pred_best_tree))
print("Recall:", recall_score(y_test, y_pred_best_tree))
print("F1 Score:", f1_score(y_test, y_pred_best_tree))
print("ROC AUC:", roc_auc_score(y_test, y_prob_best_tree))

Accuracy: 0.506514657980456
Precision: 0.5270655270655271
Recall: 0.10070767555797495
F1 Score: 0.16910420475319926
ROC AUC: 0.5049396702976386


In [36]:
rf_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_forest.feature_importances_
})

rf_importance_df = rf_importance_df.sort_values(
    "Importance",
    ascending=False
)

rf_importance_df

,Feature,Importance
3,moneyline_away,0.297169
2,moneyline_home,0.266130
0,total,0.199805
1,spread,0.152327
4,season,0.084569


## Final Comparison

In [38]:
comparison = pd.DataFrame({
    "Model": [
        f"Decision Tree (Depth={tree_cv.best_params_['max_depth']})",
        f"Random Forest (Depth={forest_cv.best_params_['max_depth']})"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_best_tree),
        accuracy_score(y_test, y_pred_best_forest)
    ],
    "Precision": [
        precision_score(y_test, y_pred_best_tree),
        precision_score(y_test, y_pred_best_forest)
    ],
    "Recall": [
        recall_score(y_test, y_pred_best_tree),
        recall_score(y_test, y_pred_best_forest)
    ],
    "F1": [
        f1_score(y_test, y_pred_best_tree),
        f1_score(y_test, y_pred_best_forest)
    ],
    "ROC AUC": [
        roc_auc_score(y_test, y_prob_best_tree),
        roc_auc_score(y_test, y_prob_best_forest)
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1,ROC AUC
0,Decision Tree (Depth=10),0.506515,0.527066,0.100708,0.169104,0.504940
1,Random Forest (Depth=15),0.520630,0.526632,0.382145,0.442902,0.523799


## Week 6 Conclusions

This week, Decision Trees and Random Forests were used to predict whether NBA games would finish Over or Under the sportsbook total. The initial unrestricted models showed substantial overfitting, with high training accuracy but poor test performance.

To address overfitting, three-fold cross-validation was used to select the maximum tree depth based on training data. Cross-validation selected a depth of 10 for the Decision Tree and 15 for the Random Forest.

The Decision Tree achieved 50.65% test accuracy and a ROC AUC of 0.5049. However, its recall was only 0.1007, showing that it struggled to correctly identify games finishing Over the sportsbook total.

The Random Forest performed better, achieving 52.06% accuracy, an F1 score of 0.4429, and a ROC AUC of 0.5238. Combining multiple trees improved overall classification performance compared with the individual Decision Tree, although the results remained close to random guessing.

Overall, this analysis demonstrated the importance of controlling model complexity and using cross-validation to select hyperparameters. Despite improvements from the Random Forest, sportsbook variables alone provided limited predictive power for classifying NBA Over/Under outcomes. Additional historical team performance features may be necessary to achieve stronger predictions.